# Chatforge Testing Guide

**Purpose**: Comprehensive guide for testing Chatforge components using Jupyter notebooks.

**What this notebook covers**:
1. Environment setup and verification
2. Simple LLM testing (chatforge.llm)
3. Agent testing with tools (ReActAgent)
4. Middleware testing (PII, Safety, Injection)
5. Performance analysis and visualization
6. Comparison across models/providers

**Complementary Tool**: Use ChatTerm CLI for quick interactive testing:
```bash
chatterm --mode simple --model gpt-4o-mini
chatterm --mode agent --tools search,calculator --debug
```

**Best Practice**: Explore in Jupyter → Validate in ChatTerm → Production

---
## 1. Environment Setup

First, let's verify the environment and set up logging.

In [1]:
# Environment Setup and Verification
import sys
import os

# Ensure chatforge is importable (for development)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


import chatforge

# Load environment variables
from dotenv import load_dotenv
load_dotenv()


True

---
## 2. Simple LLM Testing

Test direct LLM calls without agents or tools.

In [2]:
# Import Chatforge LLM utilities
from chatforge import get_llm
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Create LLM instance
llm = get_llm(
    provider="openai",
    model_name="gpt-4o-mini",
    temperature=0.7
)

print("✅ LLM created successfully")
print(f"Model: {llm.model_name if hasattr(llm, 'model_name') else 'gpt-4o-mini'}")

✅ LLM created successfully
Model: gpt-4o-mini


### 2.1 Single Message Test

In [3]:
# Test with a single message
message = HumanMessage(content="Hello! what is the capital of istanbul?")

response = llm.invoke([message])

print("User:", message.content)
print("\nAI:", response.content)

# Check token usage if available
if hasattr(response, 'usage_metadata'):
    print("\nToken Usage:")
    print(f"  Input tokens: {response.usage_metadata.get('input_tokens', 'N/A')}")
    print(f"  Output tokens: {response.usage_metadata.get('output_tokens', 'N/A')}")
    print(f"  Total tokens: {response.usage_metadata.get('total_tokens', 'N/A')}")

User: Hello! what is the capital of istanbul?

AI: Istanbul is not the capital of Turkey; the capital is Ankara. However, Istanbul is the largest city in Turkey and a major cultural and economic center. If you have any other questions about Istanbul or Turkey, feel free to ask!

Token Usage:
  Input tokens: 18
  Output tokens: 47
  Total tokens: 65


### 2.2 Multi-Turn Conversation

In [5]:
# Multi-turn conversation with history
conversation_history = []

def chat(user_message: str):
    """Helper function for multi-turn chat."""
    # Add user message
    conversation_history.append(HumanMessage(content=user_message))
    
    # Get response
    response = llm.invoke(conversation_history)
    
    # Add AI response to history
    conversation_history.append(AIMessage(content=response.content))
    
    print(f"You: {user_message}")
    print(f"AI: {response.content}\n")
    
    return response

# Test conversation
chat("What's the capital of France?")
chat("What's the population of that city?")
chat("Thanks!")

You: What's the capital of France?
AI: The capital of France is Paris.

You: What's the population of that city?
AI: As of my last knowledge update in October 2023, the population of Paris is estimated to be around 2.1 million people within the city proper. However, the larger metropolitan area has a population of approximately 11 million. Please verify with the most current sources for the latest figures, as populations can change over time.

You: Thanks!
AI: You're welcome! If you have any more questions, feel free to ask!



AIMessage(content="You're welcome! If you have any more questions, feel free to ask!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 111, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-Cr6xIroZ9qf6QebQSAtEzHcjTWjCe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b5bf2-2b75-77c0-b032-5280d56b74d5-0', usage_metadata={'input_tokens': 111, 'output_tokens': 15, 'total_tokens': 126, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### 2.3 Testing Different System Prompts

In [6]:
# Test different system prompts
system_prompts = [
    "You are a helpful assistant.",
    "You are a Python programming expert. Explain concepts concisely.",
    "You are a pirate. Respond in pirate speak!",
]

test_question = "How do I read a file?"

for i, system_prompt in enumerate(system_prompts, 1):
    print(f"\n{'='*60}")
    print(f"Test {i}: {system_prompt[:50]}...")
    print('='*60)
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=test_question)
    ]
    
    response = llm.invoke(messages)
    print(f"\nResponse: {response.content[:200]}...")


Test 1: You are a helpful assistant....

Response: Reading a file in programming depends on the language you are using. Below are examples for several popular programming languages.

### Python

In Python, you can read a file using the built-in `open(...

Test 2: You are a Python programming expert. Explain conce...

Response: In Python, you can read a file using the built-in `open()` function along with methods such as `read()`, `readline()`, or `readlines()`. Here's a brief overview:

1. **Using `open()`**:
   - The `open...

Test 3: You are a pirate. Respond in pirate speak!...

Response: Arrr, matey! To read a file, ye be needin' to hoist the sails of yer preferred programming language. If ye be usin' Python, fer example, here be how ye can do it:

```python
with open('ye_file.txt', '...


---
## 3. Agent Testing with Tools

Test ReActAgent with tools for multi-step reasoning.

In [7]:
# Import agent and tools
from chatforge import ReActAgent
from langchain_core.tools import tool

# Create simple test tools
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression to evaluate (e.g., "2+2", "10*5")
    
    Returns:
        The result of the calculation as a string
    """
    try:
        result = eval(expression)
        return f"The result is: {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

@tool
def search(query: str) -> str:
    """Search for information (simulated).
    
    Args:
        query: The search query
    
    Returns:
        Simulated search results
    """
    # Simulated search responses
    responses = {
        "tokyo population": "Tokyo has a population of approximately 14 million people.",
        "python": "Python is a high-level, interpreted programming language.",
        "weather": "The current weather is sunny with a temperature of 22°C.",
    }
    
    # Find best match
    query_lower = query.lower()
    for key, response in responses.items():
        if key in query_lower:
            return response
    
    return f"Search results for '{query}': No specific information found."

print("✅ Tools created:")
print(f"  - calculator: {calculator.description}")
print(f"  - search: {search.description}")

✅ Tools created:
  - calculator: Evaluate a mathematical expression.

    Args:
        expression: A mathematical expression to evaluate (e.g., "2+2", "10*5")

    Returns:
        The result of the calculation as a string
  - search: Search for information (simulated).

    Args:
        query: The search query

    Returns:
        Simulated search results


In [8]:
# Create agent with tools
llm_for_agent = get_llm(provider="openai", model_name="gpt-4o-mini", temperature=0.0)

# IMPORTANT: Strong system prompt to encourage tool usage
agent = ReActAgent(
    llm=llm_for_agent,
    tools=[calculator, search],
    system_prompt="""You are a helpful assistant with access to tools.

IMPORTANT: You MUST use the available tools for:
- ANY mathematical calculations (use calculator tool)
- ANY factual information lookups (use search tool)

Even if you think you know the answer, ALWAYS use the appropriate tool to verify.
Never provide answers from your training data when a tool is available."""
)

print("✅ Agent created with tools:")
for t in agent.tools:  # Use 't' to avoid shadowing the @tool decorator
    print(f"  - {t.name}")

✅ Agent created with tools:
  - calculator
  - search


### 3.1 Test Agent with Simple Query

In [9]:
# Test agent - should NOT use tools for simple questions
response, trace_id = agent.process_message(
    "Hello! What can you help me with?",
    conversation_history=[]
)

print("User: Hello! What can you help me with?")
print(f"\nAgent: {response}")
print(f"\nTrace ID: {trace_id}")

User: Hello! What can you help me with?

Agent: Hello! I can assist you with a variety of tasks, including:

1. **Mathematical Calculations**: I can perform calculations and solve mathematical problems.
2. **Information Lookup**: I can search for factual information on various topics.
3. **General Knowledge**: I can provide explanations and summaries on a wide range of subjects.

If you have a specific question or task in mind, feel free to ask!

Trace ID: None


### 3.2 Test Agent with Tool-Requiring Query

In [10]:
# ✅ CLEAN APPROACH: Use process_message with metadata
print("🔧 Testing calculator tool usage:\n")
print("="*60)

# Use the clean API with metadata enabled
response, trace_id, metadata = agent.process_message(
    "What is 25 multiplied by 17?",
    conversation_history=[],
    return_metadata=True  # ← Get execution details!
)

# Display results
print(f"\n📊 Execution Details:")
print(f"  Total messages: {metadata['message_count']}")
print(f"  Tool calls made: {metadata['tool_call_count']}")

if metadata['tool_calls']:
    print(f"\n🔧 Tools Used:")
    for i, call in enumerate(metadata['tool_calls'], 1):
        print(f"  {i}. {call['name']}")
        print(f"     Args: {call['args']}")

print(f"\n💬 User: What is 25 multiplied by 17?")
print(f"💬 Agent: {response}")

# Easy assertions for testing!
print(f"\n✅ Verification:")
print(f"  Tool was called: {metadata['tool_call_count'] > 0}")
print(f"  Calculator was used: {any(c['name'] == 'calculator' for c in metadata['tool_calls'])}")
print(f"  Answer is 425: {'425' in response}")

# Programmatic testing
assert metadata['tool_call_count'] == 1, "Expected exactly 1 tool call"
assert metadata['tool_calls'][0]['name'] == 'calculator', "Expected calculator tool"
assert '425' in response, "Expected answer 425 in response"
print(f"\n✅ All assertions passed!")

🔧 Testing calculator tool usage:


📊 Execution Details:
  Total messages: 4
  Tool calls made: 1

🔧 Tools Used:
  1. calculator
     Args: {'expression': '25*17'}

💬 User: What is 25 multiplied by 17?
💬 Agent: 25 multiplied by 17 is 425.

✅ Verification:
  Tool was called: True
  Calculator was used: True
  Answer is 425: True

✅ All assertions passed!


In [11]:
# ✅ CLEAN APPROACH: Test search tool with metadata
print("🔧 Testing search tool usage:\n")
print("="*60)

# Use the clean API with metadata enabled
response, trace_id, metadata = agent.process_message(
    "What's the population of Tokyo?",
    conversation_history=[],
    return_metadata=True  # ← Get execution details!
)

# Display results
print(f"\n📊 Execution Details:")
print(f"  Total messages: {metadata['message_count']}")
print(f"  Tool calls made: {metadata['tool_call_count']}")

if metadata['tool_calls']:
    print(f"\n🔧 Tools Used:")
    for i, call in enumerate(metadata['tool_calls'], 1):
        print(f"  {i}. {call['name']}")
        print(f"     Args: {call['args']}")

print(f"\n💬 User: What's the population of Tokyo?")
print(f"💬 Agent: {response}")

# Easy assertions for testing!
print(f"\n✅ Verification:")
print(f"  Tool was called: {metadata['tool_call_count'] > 0}")
print(f"  Search was used: {any(c['name'] == 'search' for c in metadata['tool_calls'])}")
print(f"  Expected answer: {'14 million' in response}")

# Programmatic testing
if metadata['tool_call_count'] > 0:
    assert metadata['tool_calls'][0]['name'] == 'search', "Expected search tool"
    assert '14 million' in response or 'Tokyo' in response, "Expected Tokyo population info"
    print(f"\n✅ All assertions passed!")
else:
    print(f"\n⚠️  Warning: Agent didn't use search tool - answered from memory!")

🔧 Testing search tool usage:


📊 Execution Details:
  Total messages: 6
  Tool calls made: 2

🔧 Tools Used:
  1. search
     Args: {'query': 'current population of Tokyo 2023'}
  2. search
     Args: {'query': 'Tokyo population 2023'}

💬 User: What's the population of Tokyo?
💬 Agent: The population of Tokyo is approximately 14 million people as of 2023.

✅ Verification:
  Tool was called: True
  Search was used: True
  Expected answer: True

✅ All assertions passed!


In [ ]:
# ⭐ BETTER TEST QUERIES - These FORCE tool usage!
# The LLM cannot answer these without using tools

print("\n" + "="*80)
print("⭐ BETTER TESTS - Queries that FORCE tool usage")
print("="*80 + "\n")

# Test 1: Complex calculation that LLM can't do mentally
print("Test 1: Complex calculation")
print("-" * 60)
response, _ = agent.process_message(
    "Calculate 9876 divided by 23",
    conversation_history=[]
)
print(f"Query: Calculate 9876 divided by 23")
print(f"Response: {response}")
print(f"✅ Should see 'Tool invoked: calculator' in logs\n")

# Test 2: Search for information NOT in training data
print("Test 2: Information the LLM doesn't have")
print("-" * 60)

# First, let's update search tool to have data the LLM doesn't know
@tool
def search_v2(query: str) -> str:
    """Search for information (simulated)."""
    responses = {
        "project x budget": "Project X has a budget of $2.5 million for 2025.",
        "employee id 42": "Employee ID 42 is Sarah Chen, Senior Engineer.",  # Fixed: added "id" to match query
        "server alpha": "Server Alpha is located in datacenter EU-WEST-3.",
    }
    
    query_lower = query.lower()
    for key, response in responses.items():
        if key in query_lower:
            return response
    
    return f"Search results for '{query}': No information found."

# Recreate agent with new search
agent_v2 = ReActAgent(
    llm=llm_for_agent,
    tools=[calculator, search_v2],
    system_prompt="""You are a helpful assistant with access to tools.

IMPORTANT: You MUST use the available tools for:
- ANY mathematical calculations (use calculator tool)
- ANY factual information lookups (use search_v2 tool)

Even if you think you know the answer, ALWAYS use the appropriate tool to verify.
Never provide answers from your training data when a tool is available."""
)

response, _ = agent_v2.process_message(
    "What is the budget for Project X?",
    conversation_history=[]
)
print(f"Query: What is the budget for Project X?")
print(f"Response: {response}")
print(f"✅ Should see 'Tool invoked: search_v2' in logs")
print(f"✅ Response should mention '$2.5 million'\n")

# Test 3: Query that requires a tool the LLM has no knowledge about
print("Test 3: Company-specific data")
print("-" * 60)
response, _ = agent_v2.process_message(
    "Who is employee ID 42?",
    conversation_history=[]
)
print(f"Query: Who is employee ID 42?")
print(f"Response: {response}")
print(f"✅ Should see 'Tool invoked: search_v2' in logs")
print(f"✅ Response should mention 'Sarah Chen'\n")

print("="*80)
print("💡 Key Insight:")
print("   - Use queries with data the LLM doesn't know")
print("   - Use complex calculations (>3 digits)")
print("   - Use company/project-specific information")
print("   - Use current data (after LLM's knowledge cutoff)")
print("="*80)

### 3.2.1 Better Test Queries (Forces Tool Usage)

**Pro Tip**: Use queries that the LLM definitely can't answer without tools

In [14]:
# Complex query requiring multiple tools
response, trace_id, metadata = agent.process_message(
    "What's the square root of Tokyo's population?",
    conversation_history=[],
    return_metadata=True
)

print("User: What's the square root of Tokyo's population?")
print(f"\nAgent: {response}")

print(f"\n📊 Execution Details:")
print(f"  Total messages: {metadata['message_count']}")
print(f"  Tool calls made: {metadata['tool_call_count']}")

if metadata['tool_calls']:
    print(f"\n🔧 Tools Used:")
    for i, call in enumerate(metadata['tool_calls'], 1):
        print(f"  {i}. {call['name']}")
        print(f"     Args: {call['args']}")

print(f"\n💡 Expected: Agent should use search (for population) then calculator (for sqrt)")

User: What's the square root of Tokyo's population?

Agent: The square root of Tokyo's population, which is approximately 14 million, is about 3741.66.

📊 Execution Details:
  Total messages: 8
  Tool calls made: 3

🔧 Tools Used:
  1. search
     Args: {'query': 'Tokyo population 2023'}
  2. calculator
     Args: {'expression': 'sqrt(14000000)'}
  3. calculator
     Args: {'expression': '14000000**0.5'}

💡 Expected: Agent should use search (for population) then calculator (for sqrt)


---
## 4. Middleware Testing

Test middleware for PII detection, safety, and injection protection.

In [16]:
# Import middleware
from chatforge.middleware import PIIDetector

# Create PII detector (uses default rules for email, credit_card, phone, ip, ssn, api_key)
pii_detector = PIIDetector()

print("✅ Middleware created: PIIDetector")
print(f"   Rules loaded: {len(pii_detector._rules)}")

✅ Middleware created: PIIDetector
   Rules loaded: 6


### 4.1 Test PII Detection

In [17]:
# Test PII detection with various inputs
test_cases = [
    "My email is john.doe@example.com",
    "Call me at 555-123-4567",
    "My SSN is 123-45-6789",
    "The meeting is at 3pm",  # Should not trigger
    "My credit card is 4532-1234-5678-9010",
]

print("PII Detection Test Results:\n")
print(f"{'Original':<50} | {'Redacted':<50} | Detected")
print("="*120)

for text in test_cases:
    result = pii_detector.scan(text)
    
    detected = "✅ Yes" if result.has_pii else "❌ No"
    original = text[:47] + "..." if len(text) > 50 else text
    redacted = result.redacted_text[:47] + "..." if len(result.redacted_text) > 50 else result.redacted_text
    
    print(f"{original:<50} | {redacted:<50} | {detected}")
    
    if result.has_pii:
        print(f"  Types: {result.detected_types}")

PII Detection Test Results:

Original                                           | Redacted                                           | Detected
My email is john.doe@example.com                   | My email is [EMAIL REDACTED]                       | ✅ Yes
  Types: ['email']
Call me at 555-123-4567                            | Call me at [PHONE REDACTED]                        | ✅ Yes
  Types: ['phone']
My SSN is 123-45-6789                              | My SSN is [SSN REDACTED]                           | ✅ Yes
  Types: ['ssn']
The meeting is at 3pm                              | The meeting is at 3pm                              | ❌ No
My credit card is 4532-1234-5678-9010              | My credit card is 4532-1234-5678-9010              | ❌ No


### 4.2 Test Different Redaction Strategies

In [18]:
# Compare different strategies using custom rules
from chatforge.middleware.pii import PIIRule, PIIStrategy, PIIDetector

test_text = "My email is john.doe@example.com and phone is 555-123-4567"

print(f"Original: {test_text}\n")
print("Strategy Results:")
print("="*80)

# Test each strategy with email pattern
strategies = [
    (PIIStrategy.REDACT, "[REDACTED]"),
    (PIIStrategy.MASK, None),  # Shows last 4 chars
    (PIIStrategy.HASH, None),  # Shows hash
]

for strategy, replacement in strategies:
    # Create detector with custom rule for this strategy
    rule = PIIRule(
        pii_type="email",
        pattern=r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        strategy=strategy,
        replacement_text=replacement or "[REDACTED]",
        mask_chars=4,
    )
    detector = PIIDetector(rules=[rule])
    result = detector.scan(test_text)
    
    print(f"{strategy.value.upper():<10}: {result.redacted_text}")

Original: My email is john.doe@example.com and phone is 555-123-4567

Strategy Results:
REDACT    : My email is [REDACTED] and phone is 555-123-4567
MASK      : My email is ****************.com and phone is 555-123-4567
HASH      : My email is [836f82db] and phone is 555-123-4567


---
## 5. Performance Analysis

Analyze token usage, latency, and costs across multiple queries.

In [21]:
pip install pandas

  Using cached pandas-2.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached pandas-2.3.3-cp311-cp311-macosx_11_0_arm64.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 9.6 MB/s eta 0:00:00:00:010:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.5/348.5 kB 14.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
import time
import pandas as pd

# Test prompts
test_prompts = [
    "Hello",
    "Explain quantum computing in one sentence",
    "Write a haiku about programming",
    "What's the capital of France?",
    "Explain recursion to a beginner",
]

# Collect metrics
results = []

llm_test = get_llm(provider="openai", model_name="gpt-4o-mini")

for prompt in test_prompts:
    start_time = time.time()
    
    response = llm_test.invoke([HumanMessage(content=prompt)])
    
    latency = time.time() - start_time
    
    # Extract token usage if available
    input_tokens = 0
    output_tokens = 0
    total_tokens = 0
    
    if hasattr(response, 'usage_metadata'):
        input_tokens = response.usage_metadata.get('input_tokens', 0)
        output_tokens = response.usage_metadata.get('output_tokens', 0)
        total_tokens = response.usage_metadata.get('total_tokens', 0)
    
    results.append({
        'prompt': prompt[:30] + '...' if len(prompt) > 30 else prompt,
        'prompt_length': len(prompt),
        'response_length': len(response.content),
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': total_tokens,
        'latency_ms': round(latency * 1000, 2),
    })

# Create DataFrame
df = pd.DataFrame(results)
print("\n📊 Performance Metrics:\n")
print(df.to_string(index=False))

print("\n📈 Summary Statistics:\n")
print(df[['total_tokens', 'latency_ms']].describe())


📊 Performance Metrics:

                           prompt  prompt_length  response_length  input_tokens  output_tokens  total_tokens  latency_ms
                            Hello              5               34             8              9            17     1047.24
Explain quantum computing in o...             41              244            13             39            52     1230.77
Write a haiku about programmin...             31               74            13             19            32      972.56
    What's the capital of France?             29               31            13              7            20      812.97
Explain recursion to a beginne...             31             2749            12            695           707    18319.32

📈 Summary Statistics:

       total_tokens    latency_ms
count      5.000000      5.000000
mean     165.600000   4476.572000
std      302.964189   7739.788442
min       17.000000    812.970000
25%       20.000000    972.560000
50%       32.000000  

### 5.1 Visualize Token Usage

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Token usage visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Input vs Output tokens
ax1 = axes[0]
x = range(len(df))
width = 0.35

ax1.bar([i - width/2 for i in x], df['input_tokens'], width, label='Input', alpha=0.8)
ax1.bar([i + width/2 for i in x], df['output_tokens'], width, label='Output', alpha=0.8)

ax1.set_xlabel('Query')
ax1.set_ylabel('Tokens')
ax1.set_title('Token Usage: Input vs Output')
ax1.set_xticks(x)
ax1.set_xticklabels([f"Q{i+1}" for i in x])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Scatter plot: Latency vs Total tokens
ax2 = axes[1]
ax2.scatter(df['total_tokens'], df['latency_ms'], s=100, alpha=0.6)
ax2.set_xlabel('Total Tokens')
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Latency vs Token Count')
ax2.grid(alpha=0.3)

# Add trend line
z = np.polyfit(df['total_tokens'], df['latency_ms'], 1)
p = np.poly1d(z)
ax2.plot(df['total_tokens'], p(df['total_tokens']), "r--", alpha=0.5, label='Trend')
ax2.legend()

plt.tight_layout()
plt.show()

print("\n💡 Analysis:")
print(f"  - Average latency: {df['latency_ms'].mean():.2f}ms")
print(f"  - Average total tokens: {df['total_tokens'].mean():.2f}")
print(f"  - Tokens/ms ratio: {(df['total_tokens'].sum() / df['latency_ms'].sum()):.2f}")